# 🛒 Shopper Spectrum: Customer Segmentation and Product Recommendations in E-Commerce
### *An End-to-End Unsupervised Machine Learning and Collaborative Filtering Pipeline*

---

## 📌 Project Overview
This notebook demonstrates a complete e-commerce data analytics and machine learning pipeline:
1. **Data Cleaning & Preprocessing**: Handling missing values, returns, and improper transactions.
2. **Exploratory Data Analysis (EDA)**: Insights into sales volume, top products, country distributions, and purchasing patterns.
3. **RFM Feature Engineering**: Calculating **Recency**, **Frequency**, and **Monetary** value for customer behavior modeling.
4. **Customer Segmentation via KMeans Clustering**: Scaled feature normalization, Elbow & Silhouette evaluation, and cluster profiling.
5. **Item-Based Collaborative Filtering**: Generating product-product similarity matrix using cosine distance.
6. **Model Persistence**: Exporting models and artifacts for live deployment in Streamlit.


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import warnings
warnings.filterwarnings("ignore")

print("All required libraries successfully imported.")


## 📥 Step 1: Dataset Collection & Preprocessing
We load the retail transaction dataset containing `InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `Country`, and `CustomerID`.


In [ ]:
# Load dataset
df_raw = pd.read_csv("sample_data.csv")
print(f"Raw Dataset Shape: {df_raw.shape}")
df_raw.head()


In [ ]:
# Data Preprocessing
print("Initial Missing Values:")
print(df_raw.isnull().sum())

# 1. Remove missing CustomerID
df_clean = df_raw.dropna(subset=['CustomerID']).copy()

# 2. Exclude cancelled invoices (starting with 'C')
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C', na=False)]

# 3. Remove non-positive quantities and prices
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# Format data types
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['TotalSpend'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f"\nCleaned Dataset Shape: {df_clean.shape}")
df_clean.describe()


## 📊 Step 2: Exploratory Data Analysis (EDA)
We visualize sales trends, top-selling items, and geographical distribution using Plotly.


In [ ]:
# Top 10 Selling Products
top_products = df_clean.groupby('Description')['Quantity'].sum().reset_index()
top_products = top_products.sort_values(by='Quantity', ascending=False).head(10)

fig = px.bar(
    top_products, 
    x='Quantity', 
    y='Description', 
    orientation='h',
    title='<b>Top 10 Selling Products by Quantity</b>',
    color='Quantity',
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_dark')
fig.show()


In [ ]:
# Revenue by Country
country_revenue = df_clean.groupby('Country')['TotalSpend'].sum().reset_index().sort_values(by='TotalSpend', ascending=False)

fig_country = px.bar(
    country_revenue,
    x='Country',
    y='TotalSpend',
    title='<b>Total Spend by Country</b>',
    color='TotalSpend',
    color_continuous_scale='Tealgrn'
)
fig_country.update_layout(template='plotly_dark')
fig_country.show()


## 🧠 Step 3: RFM Feature Engineering & Customer Segmentation
- **Recency**: Days since last purchase relative to snapshot date.
- **Frequency**: Total unique transaction count per customer.
- **Monetary**: Total spend per customer.


In [ ]:
# Calculate snapshot date
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# Group by CustomerID
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalSpend': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']
rfm.head()


In [ ]:
# Standardize RFM Values
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# Determine optimal K using Elbow Method & Silhouette Score
inertia = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = km.fit_predict(rfm_scaled)
    inertia.append(km.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, cluster_labels))

# Plot Elbow and Silhouette
fig_eval = go.Figure()
fig_eval.add_trace(go.Scatter(x=list(K_range), y=inertia, name='Inertia (Elbow)', mode='lines+markers'))
fig_eval.update_layout(title='<b>KMeans Elbow Curve</b>', xaxis_title='Number of Clusters (k)', yaxis_title='Inertia', template='plotly_dark')
fig_eval.show()


In [ ]:
# Train final KMeans Model with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Segment Interpretation
cluster_summary = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean()
print("Cluster Summary (Means):")
print(cluster_summary)


In [ ]:
# 3D Scatter Plot of RFM Clusters
fig_3d = px.scatter_3d(
    rfm, 
    x='Recency', 
    y='Frequency', 
    z='Monetary',
    color='Cluster',
    title='<b>3D Visualization of Customer Segments</b>',
    opacity=0.8
)
fig_3d.update_layout(template='plotly_dark')
fig_3d.show()


## 🛍️ Step 4: Item-Based Collaborative Filtering Recommendation Engine
We construct a **CustomerID x Product Description** matrix and compute product-product cosine similarity scores.


In [ ]:
# Pivot Table
user_item = df_clean.pivot_table(index='CustomerID', columns='Description', values='Quantity', aggfunc='sum', fill_value=0)

# Cosine Similarity Matrix
item_similarity = cosine_similarity(user_item.T)
similarity_df = pd.DataFrame(item_similarity, index=user_item.columns, columns=user_item.columns)

# Recommendation function test
def recommend_products(product_name, top_n=5):
    if product_name not in similarity_df.columns:
        return []
    scores = similarity_df[product_name].sort_values(ascending=False)[1:top_n+1]
    return list(zip(scores.index, scores.values))

test_prod = similarity_df.columns[0]
print(f"Sample Recommendations for '{test_prod}':")
for p, s in recommend_products(test_prod):
    print(f" - {p} (Similarity: {s:.2%})")


## 💾 Step 5: Model Artifact Exporting
Save trained models and matrices for deployment into Streamlit app.


In [ ]:
# Save artifacts using joblib
joblib.dump(kmeans, "kmeans_model.pkl")
joblib.dump(scaler, "rfm_scaler.pkl")
joblib.dump(similarity_df, "similarity_matrix.pkl")

print("Artifacts successfully saved: kmeans_model.pkl, rfm_scaler.pkl, similarity_matrix.pkl")
